# SI4006 · Sesión 10 · Extra: seguimiento de evaluaciones con Weights & Biases

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 3 · RAG

Este notebook acompaña la slide de W&B. En la S08 y la S10 ya generamos números con el harness y con
RAGAS. El problema es que esos números se pierden si solo los miramos en pantalla. Weights & Biases es
la herramienta de seguimiento del curso: registramos cada corrida (sus puntajes, la latencia y, si
queremos, la traza del agente) y así comparamos versiones de nuestro sistema en el tiempo.

> Corre **offline**, sin cuenta ni clave. Al final se explica cómo pasarlo a online con una cuenta
> gratuita. En la S15 retomamos W&B, ya para observar el sistema desplegado en producción.

## 0 · Setup

In [ ]:
%pip install -q wandb
import os, wandb
# Offline: escribe las corridas en disco, sin necesidad de cuenta ni de internet.
# Para online, comenten esta línea y corran `wandb login` con su cuenta gratuita.
os.environ['WANDB_MODE'] = 'offline'
print('wandb', wandb.__version__, '| modo:', os.environ['WANDB_MODE'])

## 1 · Tres versiones del sistema, tres corridas

Imaginemos que evaluamos tres versiones de nuestro RAG con el mismo eval set: el ingenuo (S07), el
avanzado (S08) y el agéntico (S10). Cada versión tiene sus puntajes del harness de M2 y sus métricas
de RAGAS, más la latencia. Registramos cada una como una corrida en W&B.

**Reemplacen estos números por los de su sistema real.** Aquí son ilustrativos.

In [ ]:
# Cada versión: puntajes del harness (M2) + métricas de RAGAS + latencia por consulta.
versiones = {
    'rag_ingenuo':  dict(sim_embeddings=0.71, llm_juez=3.4, aciertos_frac=6/12, latencia_s=1.2,
                         faithfulness=0.72, context_precision=0.55, context_recall=0.60, answer_relevancy=0.78),
    'rag_avanzado': dict(sim_embeddings=0.78, llm_juez=3.9, aciertos_frac=9/12, latencia_s=2.1,
                         faithfulness=0.81, context_precision=0.74, context_recall=0.79, answer_relevancy=0.83),
    'rag_agentico': dict(sim_embeddings=0.80, llm_juez=4.1, aciertos_frac=10/12, latencia_s=4.6,
                         faithfulness=0.88, context_precision=0.76, context_recall=0.81, answer_relevancy=0.85),
}

for nombre, metricas in versiones.items():
    run = wandb.init(project='si4006-s10-rag', name=nombre, config={'version': nombre}, reinit=True)
    wandb.log(metricas)          # registra los puntajes de esta corrida
    wandb.finish()
    print('registrada la corrida:', nombre)

## 2 · Una tabla comparativa

Además de cada corrida por separado, guardamos una tabla que las pone lado a lado. En el panel de W&B
esta tabla es interactiva; aquí queda registrada en la corrida.

In [ ]:
run = wandb.init(project='si4006-s10-rag', name='comparativa', reinit=True)
cols = ['version', 'sim_embeddings', 'llm_juez', 'aciertos_frac', 'latencia_s',
        'faithfulness', 'context_precision', 'context_recall', 'answer_relevancy']
tabla = wandb.Table(columns=cols)
for nombre, m in versiones.items():
    tabla.add_data(nombre, m['sim_embeddings'], m['llm_juez'], round(m['aciertos_frac'],2), m['latencia_s'],
                   m['faithfulness'], m['context_precision'], m['context_recall'], m['answer_relevancy'])
wandb.log({'comparativa': tabla})
wandb.finish()
print('tabla comparativa registrada')

## 3 · La traza de un agente

Cuando el sistema es un agente (S10), conviene registrar también qué hizo en cada paso: su pensamiento,
la herramienta que llamó y la observación que recibió. Así, si una respuesta salió mal, podemos ver en
qué paso se torció.

In [ ]:
traza = [
    (1, 'Necesito el reglamento sobre tareas', 'buscar_en_corpus("plazo de tareas")', 'Art. 12: hasta 15%'),
    (2, 'Ya tengo la regla, ahora el cálculo', 'calculadora("0.15 * 40")', '6.0'),
    (3, 'Puedo responder', 'responder', 'La penalización máxima es 6 puntos.'),
]
run = wandb.init(project='si4006-s10-rag', name='traza_agente', reinit=True)
t = wandb.Table(columns=['paso', 'pensamiento', 'accion', 'observacion'])
for fila in traza:
    t.add_data(*fila)
wandb.log({'traza_agente': t})
wandb.finish()
print('traza registrada')

## 4 · Cómo verlo, y cómo pasarlo a online

En modo offline, W&B guarda todo en la carpeta `wandb/` de la sesión. Para verlo en el panel web más
tarde, se sube con `wandb sync wandb/offline-run-...`.

Para trabajar online desde el inicio (recomendado para el proyecto): creen una cuenta gratuita en
wandb.ai, corran `wandb login` (pega su clave) y quiten la línea `WANDB_MODE = 'offline'` del setup.
Cada corrida aparece en el panel, con gráficos y tablas comparables entre sí.

**Para M3 es opcional**, pero registrar sus corridas desde ahora les evita perder resultados y es la
base de la observabilidad que veremos en la S15.

In [ ]:
print('Corridas guardadas en:', os.path.abspath('wandb'))
for f in sorted(os.listdir('wandb'))[:10]:
    print('  ', f)